In [ ]:
from google.colab import files
import pandas as pd

# Upload file
uploaded = files.upload()
filename = next(iter(uploaded))

# Try reading with ISO-8859-1 encoding
df = pd.read_csv(filename, encoding='ISO-8859-1')


Saving flipkart_product.csv to flipkart_product.csv


In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AlbertTokenizer, AlbertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files
import shutil

# 🧹 Combine Review and Summary into one text column
df["text"] = df["Review"].astype(str) + " " + df["Summary"].astype(str)

# 🧼 Clean encoding issues and missing values
df["text"] = df["text"].str.encode('utf-8', errors='ignore').str.decode('utf-8')
df.dropna(subset=["text", "Rate"], inplace=True)

# 🔢 Convert Rate to numeric safely
df["Rate"] = pd.to_numeric(df["Rate"], errors="coerce")

# 🏷️ Convert Rate to label (1 = Positive, 0 = Negative, drop Neutral)
df["label"] = df["Rate"].apply(lambda x: 1 if x >= 4 else (0 if x <= 2 else None))
df.dropna(subset=["label"], inplace=True)
df["label"] = df["label"].astype(int)

# 🧪 Split into train/test sets
train_df, test_df = train_test_split(df[["text", "label"]], test_size=0.2, random_state=42)

# 🤗 Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# 🔠 Load tokenizer and model for ALBERT
tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")
model = AlbertForSequenceClassification.from_pretrained("albert-base-v2", num_labels=2)

# ✂️ Tokenization function with reduced max_length
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=64)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# 🚫 Remove unnecessary columns
columns_to_remove = [col for col in ["text", "__index_level_0__"] if col in train_dataset.column_names]
train_dataset = train_dataset.remove_columns(columns_to_remove)

columns_to_remove_test = [col for col in ["text", "__index_level_0__"] if col in test_dataset.column_names]
test_dataset = test_dataset.remove_columns(columns_to_remove_test)

# ⚙️ Set PyTorch format
train_dataset.set_format("torch")
test_dataset.set_format("torch")

# 📦 Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 📈 Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

# Set training arguments with lower batch size and fewer epochs for faster training
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,  # Smaller batch size
    per_device_eval_batch_size=4,   # Smaller batch size
    num_train_epochs=1,  # Fewer epochs
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
)

# 🏋️‍♂️ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 🚀 Train the model
trainer.train()

# 💾 Save model and tokenizer
save_path = "./saved_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(f"{save_path}/tokenizer")

# 🧳 Zip the saved model directory for easy download
shutil.make_archive(save_path, 'zip', save_path)

# 📤 Download the zipped model to your local system
files.download(f"{save_path}.zip")

print("✅ Model training complete and saved in `saved_model/`")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: swarangisawant2702 (swarangisawant2702-abc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.151100
1000,0.102200
1500,0.070600
2000,0.058700
2500,0.082900
3000,0.069100
3500,0.079900
4000,0.057400
4500,0.059700
5000,0.042800


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model training complete and saved in `saved_model/`


In [ ]:
files.download(f"{save_path}.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>